In [1]:
import sys
sys.path.append("../") 
from libs.cna_utils import *
# For reproducibility
np.random.seed(0) 

# RESPONSE

## Single Cell

In [2]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/age_sex_response_race/sc_meta_response.csv')
print(meta.shape)

(19609, 37)


In [3]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/age_sex_response_race/sc_harmony_response.csv')
print(harmony.shape)

(19609, 20)


In [4]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/age_sex_response_race/sc_umap_response.csv')
print(umap.shape)

(19609, 2)


In [8]:
# Start by making anndata object
d = mad.MultiAnnData(X=harmony, obs=meta, sampleid="sample")
#d.obs_to_sample(['Type', 'Site',
#                 'Responder_Status', 'Sex',
#                 'Age', 'Race',
#                'Ethnicity', 'ISN',
#               'Activity', "Chronicity",
#                 'processing_batch'])
d.obs_to_sample(['Sex', 'Age', 'Final_Chronicity', 'Final_Activity', 'First_biop', 'Pred_use', 'Responder_Status', 'Race_[A]',
       'Race_[A][B]', 'Race_[B]', 'Race_[B][AI]',  'Race_[U]',
       'Race_[W]', 'Final_ISN_[III]', 'Final_ISN_[III][V]', 'Final_ISN_[IV]',
       'Final_ISN_[IV][V]', 'Final_ISN_[V]', 
       'Final_Site_Einstein', 'Final_Site_JHU', 'Final_Site_Michigan',
       'Final_Site_MUSC', 'Final_Site_Northwell', 'Final_Site_NYU',
       'Final_Site_Rochester', 'Final_Site_Texas Tech', 
       'Final_Site_UCSD', 'Final_Site_UCSF', 'injured_pt_prop'])
d.samplem.head()

['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,Sex,Age,Final_Chronicity,Final_Activity,First_biop,Pred_use,Responder_Status,Race_[A],Race_[A][B],Race_[B],...,Final_Site_JHU,Final_Site_Michigan,Final_Site_MUSC,Final_Site_Northwell,Final_Site_NYU,Final_Site_Rochester,Final_Site_Texas Tech,Final_Site_UCSD,Final_Site_UCSF,injured_pt_prop
sample,,,,,,,,,,,,,,,,,,,,,
AMPSLEkid_cells_0134,1.0,1.028902,6.0,4.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,16.551724
AMPSLEkid_cells_0137,1.0,0.078878,3.0,5.0,0.0,1.0,2.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,14.341498
AMPSLEkid_cells_0138,1.0,0.597073,0.0,0.0,1.0,1.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,5.629663
AMPSLEkid_cells_0139,1.0,0.942537,3.0,2.0,0.0,1.0,2.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,4.405520
AMPSLEkid_cells_0140,1.0,-0.698414,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,6.399412


In [9]:
umap.index = d.obs.index
d.obsm['X_umap'] = umap

In [10]:
np.random.seed(0) 
cna.pp.knn(d)

computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Responder_Status, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 41.257476806640625
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 27.17543601989746
	20th percentile R2(t,t-1): 0.7784479260444641
	taking step 3
	median kurtosis: 19.716590881347656
	20th percentile R2(t,t-1): 0.9197142243385314
	taking step 4
	median kurtosis: 15.091355323791504
	20th percentile R2(t,t-1): 0.9542247772216796
	taking step 5
	median kurtosis: 12.27673053741455
	20th percentile R2(t,t-1): 0.9710082650184632
stopping after 5 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
19
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.06239376062393761 , 19 PCs used
total r^2 between top 19 NAM PCs and outcome is 0.25


In [33]:
sc_uni['Responder_Status'] = res.p

In [13]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.injured_pt_prop, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      covs = d.samplem[['Responder_Status']], 
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_association.py:57: UserWarning: data supported use of 20 NAM PCs, which is the maximum considered. Consider allowing more PCs by using the "ks" argument.
  warnings.warn(('data supported use of {} NAM PCs, which is the maximum considered. '+\
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_association.py:74: UserWarning: global association p-value attained minimal possible value. Consider increasing Nnull
  warnings.warn('global association p-value attained minimal possible value. '+\


computing neighborhood-level FDRs
20
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 9.999000099990002e-05 , 20 PCs used
total r^2 between top 20 NAM PCs and outcome is 0.43


# Case/Control

## Single Cell

In [3]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/case_control/sc_meta.csv')
print(meta.shape)

(24084, 32)


In [4]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/case_control/sc_harmony.csv')
print(harmony.shape)

(24084, 20)


In [5]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/case_control/sc_umap.csv')
print(umap.shape)

(24084, 2)


In [6]:
res = cna_test(meta, harmony, umap, "Type_numeric", covars = None)

(24084, 32)
(24084, 20)
(24084, 2)
['cell' 'sample' 'Annot.separate' 'dataset' 'Site' 'broad.type'
 'doublet_classification' 'annotation' 'final_annotation' 'kid_sample'
 'Final_Site' 'Sex' 'Type']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 64.7245101928711
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 43.574012756347656
	20th percentile R2(t,t-1): 0.7702297449111939
	taking step 3
	median kurtosis: 32.424171447753906
	20th percentile R2(t,t-1): 0.9175893902778626
	taking step 4
	median kurtosis: 25.66002655029297
	20th percentile R2(t,t-1): 0.9520996809005737
	taking step 5
	median kurtosis: 21.076290130615234
	20th percentile R2(t,t-1): 0.9691211819648743
	taking step 6
	median kurtosis: 18.137849807739258
	20th percentile R2(t,t-1): 0.9793549537658691
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_association.py:74: UserWarning: global association p-value attained minimal possible value. Consider increasing Nnull
  warnings.warn('global association p-value attained minimal possible value. '+\


computing neighborhood-level FDRs
16
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 9.999000099990002e-05 , 16 PCs used
total r^2 between top 16 NAM PCs and outcome is 0.51


In [7]:
np.savetxt('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/case_control/sc_ncorr.csv', res.ncorrs, delimiter=",")
np.savetxt('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/case_control/sc_fdrs.csv', res.fdrs, delimiter=",")

# Chronicity

## Single Cell

In [5]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sc_meta.csv')
print(meta.shape)

(21479, 39)


In [6]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sc_harmony.csv')
print(harmony.shape)

(21479, 20)


In [7]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sc_umap.csv')
print(umap.shape)

(21479, 2)


In [8]:
cna_object, res = cna_test(meta,harmony, umap, 'Final_Chronicity', covars = ['First_biop', 'Responder_Status'])

(21479, 39)
(21479, 20)
(21479, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 46.802528381347656
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 31.071922302246094
	20th percentile R2(t,t-1): 0.775562047958374
	taking step 3
	median kurtosis: 22.631181716918945
	20th percentile R2(t,t-1): 0.9188318848609924
	taking step 4
	median kurtosis: 17.426891326904297
	20th percentile R2(t,t-1): 0.9540320038795471
	taking step 5
	median kurtosis: 14.204331398010254
	20th percentile R2(t,t-1): 0.9702481627464294
	taking step 6
	median kurtosis: 12.118790626525879
	20th percentile R2(t,t-1): 0.9805166721343994
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
8
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.00019998000199980003 , 8 PCs used
total r^2 between top 8 NAM PCs and outcome is 0.30


In [9]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sc_conditional_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sc_conditional_fdrs.csv", 
               res.fdrs, delimiter=",")

In [12]:
y_resid_hat = pd.DataFrame([pd.DataFrame(cna_object.uns['NAM_sampleXpc']).index.tolist(), res.yresid_hat],
             index = ['sample', 'Predicted Chronicity']).T
y_resid_hat.to_csv("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sc_conditional_yresid_hat.csv",
                   index = False)

## Single Nuclei

In [166]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sn_meta.csv')
print(meta.shape)

(1310, 11)


In [167]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sn_harmony.csv')
print(harmony.shape)

(1310, 20)


In [168]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sn_umap.csv')
print(umap.shape)

(1310, 2)


In [169]:
# Start by making anndata object
d = mad.MultiAnnData(X=harmony, obs=meta, sampleid="sample")
#d.obs_to_sample(['Type', 'Site',
#                 'Responder_Status', 'Sex',
#                 'Age', 'Race',
#                'Ethnicity', 'ISN',
#               'Activity', "Chronicity",
#                 'processing_batch'])
d.obs_to_sample(['Final_Chronicity'])
d.samplem.head()

['cell' 'Sex' 'sample' 'final_annotation' 'Responder.Status' 'Race'
 'Final_ISN' 'Type']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,Final_Chronicity
sample,
AMPSLEkid_cells_0137,3.0
AMPSLEkid_cells_0138,0.0
AMPSLEkid_cells_0139,3.0
AMPSLEkid_cells_0147,9.0
AMPSLEkid_cells_0366,7.0


In [170]:
umap.index = d.obs.index
d.obsm['X_umap'] = umap

In [171]:
np.random.seed(0) 
cna.pp.knn(d)

computing default knn graph


In [172]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Final_Chronicity, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 8.452367078580561
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 6.326636363641104
	20th percentile R2(t,t-1): 0.755109441280365
	taking step 3
	median kurtosis: 5.628824795997255
	20th percentile R2(t,t-1): 0.9261321544647216
stopping after 3 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
4
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.30286971302869714 , 4 PCs used
total r^2 between top 4 NAM PCs and outcome is 0.35


In [97]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sn_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/chronicity/sn_fdrs.csv", 
               res.fdrs, delimiter=",")

# Activity

## Single Cell

In [2]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sc_meta.csv')
print(meta.shape)

(21479, 39)


In [3]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sc_harmony.csv')
print(harmony.shape)

(21479, 20)


In [4]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sc_umap.csv')
print(umap.shape)

(21479, 2)


In [6]:
res = cna_test(meta, harmony, umap, 'Final_Activity')

(21479, 39)
(21479, 20)
(21479, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 46.802528381347656
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 31.071922302246094
	20th percentile R2(t,t-1): 0.775562047958374
	taking step 3
	median kurtosis: 22.631181716918945
	20th percentile R2(t,t-1): 0.9188318848609924
	taking step 4
	median kurtosis: 17.426891326904297
	20th percentile R2(t,t-1): 0.9540320038795471
	taking step 5
	median kurtosis: 14.204331398010254
	20th percentile R2(t,t-1): 0.9702481627464294
	taking step 6
	median kurtosis: 12.118790626525879
	20th percentile R2(t,t-1): 0.9805166721343994
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_association.py:74: UserWarning: global association p-value attained minimal possible value. Consider increasing Nnull
  warnings.warn('global association p-value attained minimal possible value. '+\


computing neighborhood-level FDRs
3
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 9.999000099990002e-05 , 3 PCs used
total r^2 between top 3 NAM PCs and outcome is 0.35


In [11]:
activity_uni = res.p

In [7]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sc_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sc_fdrs.csv", 
               res.fdrs, delimiter=",")

In [5]:
res = cna_test(meta,harmony, umap, 'Final_Activity', 
               covars = ['Pred_use', 'Final_ISN_[IV]',
                         'Final_ISN_[IV][V]',
                         'Final_ISN_[V]',
                         'Final_Site_JHU',
                         'Final_Site_NYU'])

(21479, 39)
(21479, 20)
(21479, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 46.802528381347656
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 31.071941375732422
	20th percentile R2(t,t-1): 0.7755601406097412
	taking step 3
	median kurtosis: 22.631181716918945
	20th percentile R2(t,t-1): 0.9188318848609924
	taking step 4
	median kurtosis: 17.426891326904297
	20th percentile R2(t,t-1): 0.9540318846702576
	taking step 5
	median kurtosis: 14.204333305358887
	20th percentile R2(t,t-1): 0.9702481627464294
	taking step 6
	median kurtosis: 12.118790626525879
	20th percentile R2(t,t-1): 0.9805172681808472
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
18
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.0005999400059994001 , 18 PCs used
total r^2 between top 18 NAM PCs and outcome is 0.31


In [6]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sc_cond_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sc_cond_fdrs.csv", 
               res.fdrs, delimiter=",")

## Single Nuclei

In [106]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sn_meta.csv')
print(meta.shape)

(1590, 11)


In [107]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sn_harmony.csv')
print(harmony.shape)

(1590, 20)


In [108]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sn_umap.csv')
print(umap.shape)

(1590, 2)


In [109]:
# Start by making anndata object
d = mad.MultiAnnData(X=harmony, obs=meta, sampleid="sample")
#d.obs_to_sample(['Type', 'Site',
#                 'Responder_Status', 'Sex',
#                 'Age', 'Race',
#                'Ethnicity', 'ISN',
#               'Activity', "Chronicity",
#                 'processing_batch'])
d.obs_to_sample(['Final_Activity'])
d.samplem.head()

['cell' 'Sex' 'sample' 'final_annotation' 'Responder.Status' 'Race'
 'Final_ISN' 'Type']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,Final_Activity
sample,
AMPSLEkid_cells_0137,5.000000
AMPSLEkid_cells_0138,0.000000
AMPSLEkid_cells_0139,2.000000
AMPSLEkid_cells_0147,1.000000
AMPSLEkid_cells_0366,6.333333


In [110]:
umap.index = d.obs.index
d.obsm['X_umap'] = umap

In [111]:
np.random.seed(0) 
cna.pp.knn(d)

computing default knn graph


In [112]:
np.random.seed(0) 
res = cna.tl._association.association(d, #dataset 
                                      d.samplem.Final_Activity, #phenotype
#                                       batches=d.samplem.processing_batch, #batches
                                      Nnull=10000, # number of null permutations to do (defaults to only 1e3)
                                      ks=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] # I asked the method to consider up to 10 PCs because
                                                                #it chose the max number of PCs it considered the default set of [1,2,3,4]
                                     )
print(res.k)
print(res.ks)
print('p =', res.p, ',', res.k, 'PCs used')
print('total r^2 between top {} NAM PCs and outcome is {:.2f}'.format(res.k, res.r2))

qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 9.495479074247964
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 7.22098166948354
	20th percentile R2(t,t-1): 0.7603078842163086
	taking step 3
	median kurtosis: 6.4655592353747995
	20th percentile R2(t,t-1): 0.9297066450119018
stopping after 3 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
2
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.42925707429257076 , 2 PCs used
total r^2 between top 2 NAM PCs and outcome is 0.23


In [113]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sn_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/activity/sn_fdrs.csv", 
               res.fdrs, delimiter=",")

# ISN

## Single Cell

In [5]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/sc_meta.csv')
print(meta.shape)

(23491, 39)


In [6]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/sc_harmony.csv')
print(harmony.shape)

(23491, 20)


In [7]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/sc_umap.csv')
print(umap.shape)

(23491, 2)


In [14]:
res = cna_test(meta, harmony, umap, 'Final_ISN_[III]')

(23491, 39)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
19
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.0765923407659234 , 19 PCs used
total r^2 between top 19 NAM PCs and outcome is 0.20


In [15]:
res = cna_test(meta, harmony, umap, 'Final_ISN_[III][V]')

(23491, 39)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
4
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.967003299670033 , 4 PCs used
total r^2 between top 4 NAM PCs and outcome is 0.02


In [16]:
res = cna_test(meta, harmony, umap, 'Final_ISN_[IV]')

(23491, 39)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
1
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.0017998200179982 , 1 PCs used
total r^2 between top 1 NAM PCs and outcome is 0.10


In [79]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/sc_ISN_[IV]_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/sc_ISN_[IV]_fdrs.csv", 
               res.fdrs, delimiter=",")

In [17]:
res = cna_test(meta, harmony, umap, 
               'Final_ISN_[IV]', covars = ['Final_Activity'])

(23491, 39)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing neighborhood-level FDRs
2
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.8249175082491751 , 2 PCs used
total r^2 between top 2 NAM PCs and outcome is 0.01


In [18]:
res = cna_test(meta, harmony, umap, 'Final_ISN_[IV][V]')

(23491, 39)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
2
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.022397760223977603 , 2 PCs used
total r^2 between top 2 NAM PCs and outcome is 0.08


In [19]:
res = cna_test(meta, harmony, umap, 
               'Final_ISN_[IV][V]', 
               covars  = ['Final_Activity'])

(23491, 39)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
12
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.12038796120387961 , 12 PCs used
total r^2 between top 12 NAM PCs and outcome is 0.16


In [20]:
res = cna_test(meta, harmony, umap, 'Final_ISN_[V]')

(23491, 39)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving
computing SVD
performing association test


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_association.py:74: UserWarning: global association p-value attained minimal possible value. Consider increasing Nnull
  warnings.warn('global association p-value attained minimal possible value. '+\


computing neighborhood-level FDRs
1
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 9.999000099990002e-05 , 1 PCs used
total r^2 between top 1 NAM PCs and outcome is 0.22


In [8]:
res = cna_test(meta, harmony, umap, 'Final_ISN_[V]', 
               covars = ['Final_Activity', 
                         'Pred_use',
                         'Race_[A]',
                         'Race_[B]',
                         'Race_[W]',
                         'First_biop',
                         'Final_Site_JHU',
                         'Final_Site_NYU'])

(23491, 39)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
17
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.0037996200379962005 , 17 PCs used
total r^2 between top 17 NAM PCs and outcome is 0.28


In [9]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/cond_sc_ISN_[V]_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/cond_sc_ISN_[V]_fdrs.csv", 
               res.fdrs, delimiter=",")

## Single Nuclei

In [129]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/sn_meta.csv')
print(meta.shape)

(1767, 16)


In [130]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/sn_harmony.csv')
print(harmony.shape)

(1767, 20)


In [131]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/ISN/sn_umap.csv')
print(umap.shape)

(1767, 2)


In [132]:
# Start by making anndata object
d = mad.MultiAnnData(X=harmony, obs=meta, sampleid="sample")
#d.obs_to_sample(['Type', 'Site',
#                 'Responder_Status', 'Sex',
#                 'Age', 'Race',
#                'Ethnicity', 'ISN',
#               'Activity', "Chronicity",
#                 'processing_batch'])
d.obs_to_sample(['Final_ISN_[III]', 'Final_ISN_[III][V]', 'Final_ISN_[IV]',
                 'Final_ISN_[IV][V]', 'Final_ISN_[V]'])
d.samplem.head()

['cell' 'Sex' 'sample' 'final_annotation' 'Responder.Status' 'Race'
 'Final_ISN' 'Type']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,Final_ISN_[III],Final_ISN_[III][V],Final_ISN_[IV],Final_ISN_[IV][V],Final_ISN_[V]
sample,,,,,
AMPSLEkid_cells_0137,1.0,0.000000,0.0,0.000000,0.0
AMPSLEkid_cells_0138,0.0,1.000000,0.0,0.000000,0.0
AMPSLEkid_cells_0139,0.0,1.000000,0.0,0.000000,0.0
AMPSLEkid_cells_0147,1.0,0.000000,0.0,0.000000,0.0
AMPSLEkid_cells_0366,0.0,0.333333,0.0,0.666667,0.0


In [143]:
pd.DataFrame.from_dict(sc_uni, orient='index').T.to_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/univariate_cna.csv')

In [153]:
sc_cond

{'Site': 0.13878612138786123,
 'Race': 0.0163983601639836,
 'Prednisone_use': 0.07219278072192781,
 'ISN': 0.0004999500049995,
 'First_biopsy': 0.051094890510948905,
 'Chronicity': 0.00019998000199980003,
 'Activity': 0.0006999300069993001,
 'Sex': 'NA',
 'Responder_Status': 'NA',
 'Age': 'NA'}

In [155]:
sc_cond['Sex'] = "NA"
sc_cond["Responder_Status"] = "NA"
sc_cond["Age"] = "NA"
pd.DataFrame.from_dict(sc_cond, orient='index').T.to_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/conditional_cna.csv')

In [50]:
sc_chron['Sex'] = "NA"
sc_chron["Responder_Status"] = "NA"
sc_chron["Age"] = "NA"
sc_chron["Prednisone_use"] = "NA"
sc_chron["First_biopsy"] = "NA"
sc_chron["Chronicity"] = "NA"
sc_chron["Site"] = "NA"

In [52]:
pd.DataFrame.from_dict(sc_chron, orient='index').T.to_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/conditional_chronicity_cna.csv')

# INJURED PT

## Single Cell

In [2]:
meta = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/injured_pt/sc_meta.csv')
print(meta.shape)

(23491, 40)


In [3]:
harmony = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/injured_pt/sc_harmony.csv')
print(harmony.shape)

(23491, 20)


In [4]:
umap = pd.read_csv('/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/injured_pt/sc_umap.csv')
print(umap.shape)

(23491, 2)


In [5]:
res = cna_test(meta, harmony, umap, 'injured_pt_prop', 
                                      covars = ['Final_Site_JHU',
                                      'Final_Site_NYU',
                                      'First_biop',
                                      'Responder_Status',
                                      'Age'])

(23491, 40)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')


computing SVD
performing association test
computing neighborhood-level FDRs
6
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.00019998000199980003 , 6 PCs used
total r^2 between top 6 NAM PCs and outcome is 0.30


In [7]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/injured_pt/sc_ncorr_cond_nochron.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/injured_pt/sc_fdrs_cond_nochron.csv", 
               res.fdrs, delimiter=",")

In [5]:
cna_object, res = cna_test(meta, 
                           harmony,
                           umap, 
                           'injured_pt_prop', 
                            covars = ['Final_Site_JHU',
                                      'Final_Site_NYU',
                                      'First_biop',
                                      'Responder_Status',
                                      'Age',
                                      'Final_Chronicity'])

(23491, 40)
(23491, 20)
(23491, 2)
['cell' 'sample' 'final_annotation' 'Responder.Status' 'Race' 'Final_ISN'
 'Type' 'Final_Site']
consider casting to numeric types where appropriate, and
consider re-coding text-valued columns with pandas.get_dummies
computing default knn graph


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/multianndata/core.py:17: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  super().__init__(*args, **kwargs)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qcd NAM not found; computing and saving
	taking step 1
	median kurtosis: 51.37150192260742
	20th percentile R2(t,t-1): nan
	taking step 2
	median kurtosis: 34.00757598876953
	20th percentile R2(t,t-1): 0.7737238287925721
	taking step 3
	median kurtosis: 24.5955810546875
	20th percentile R2(t,t-1): 0.9182487368583679
	taking step 4
	median kurtosis: 18.904521942138672
	20th percentile R2(t,t-1): 0.9531392097473145
	taking step 5
	median kurtosis: 15.179661750793457
	20th percentile R2(t,t-1): 0.9696611166000366
	taking step 6
	median kurtosis: 12.74237060546875
	20th percentile R2(t,t-1): 0.9801829218864441
stopping after 6 steps
covariate-adjusted NAM not found; computing and saving


/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:79: UserWarning: only one unique batch supplied to qc
  warnings.warn('only one unique batch supplied to qc')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_nam.py:101: UserWarning: only one unique batch supplied to prep
  warnings.warn('only one unique batch supplied to prep')
/PHShome/ssg34/.conda/envs/plswork/lib/python3.9/site-packages/cna/tools/_association.py:57: UserWarning: data supported use of 20 NAM PCs, which is the maximum considered. Consider allowing more PCs by using the "ks" argument.
  warnings.warn(('data supported use of {} NAM PCs, which is the maximum considered. '+\


computing SVD
performing association test
computing neighborhood-level FDRs
20
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
p = 0.000999900009999 , 20 PCs used
total r^2 between top 20 NAM PCs and outcome is 0.39


In [6]:
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/injured_pt/sc_cond_ncorr.csv", 
               res.ncorrs, delimiter=",")
np.savetxt("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/injured_pt/sc_cond_fdrs.csv", 
               res.fdrs, delimiter=",")

In [7]:
y_resid_hat = pd.DataFrame([pd.DataFrame(cna_object.uns['NAM_sampleXpc']).index.tolist(), res.yresid_hat],
             index = ['Unified_Visit', 'Predicted Chronicity']).T
y_resid_hat.to_csv("/data/srlab/ssg34/SLE_kidney_v2/data/cna_new/myeloid/injured_pt/cond_y_resid_hat.csv",
                   index = False)